# Phase gradient autofocus

In [ ]:
import torch
import torchbp
import matplotlib.pyplot as plt
from scipy.signal import get_window
import numpy as np

if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
print("Device:", device)

Generate synthetic radar data

In [ ]:
nr = 100 # Range points
ntheta = 128 # Azimuth points
nsweeps = 128 # Number of measurements
fc = 6e9 # RF center frequency
bw = 100e6 # RF bandwidth
tsweep = 100e-6 # Sweep length
fs = 1e6 # Sampling frequency
nsamples = int(fs * tsweep) # Time domain samples per sweep

# Imaging grid definition. Azimuth angle "theta" is sine of radians. 0.2 = 11.5 degrees.
grid_polar = {"r": (90, 110), "theta": (-0.2, 0.2), "nr": nr, "ntheta": ntheta}

In [ ]:
target_pos = torch.tensor([[100, 0, 0], [105, 10, 0], [97, -5, 0], [102, -10, 0], [95, 5, 0]], dtype=torch.float32, device=device)
target_rcs = torch.tensor([1,1,1,1,1], dtype=torch.float32, device=device)
pos = torch.zeros([nsweeps, 3], dtype=torch.float32, device=device)
pos[:,1] = torch.linspace(-nsweeps/2, nsweeps/2, nsweeps) * 0.25 * 3e8 / fc

In [ ]:
# Oversampling input data decreases interpolation errors
oversample = 3

# Modulation frequency in range direction to center the spectrum at DC
# for more accurate interpolation.
data_fmod = -torch.pi * (1 - (oversample-1) / oversample)

data = torchbp.util.generate_fmcw_data(target_pos, target_rcs, pos, fc, bw, tsweep, fs)
# Apply windowing function in range direction
wr = torch.tensor(get_window(("taylor", 3, 30), data.shape[-1])[None,:], dtype=torch.float32, device=device)
wa = torch.tensor(get_window(("taylor", 3, 30), data.shape[0])[:,None], dtype=torch.float32, device=device)
data = torch.fft.ifft(data * wa * wr, dim=-1, n=nsamples * oversample)

data_fmod_f = torch.exp(1j*data_fmod*torch.arange(data.shape[-1], device=device))[None,:]
data = data * data_fmod_f

data_db = 20*torch.log10(torch.abs(data)).detach()
m = torch.max(data_db)

plt.figure()
plt.imshow(data_db.cpu().numpy(), origin="lower", vmin=m-30, vmax=m, aspect="auto")
plt.xlabel("Range samples")
plt.ylabel("Azimuth samples");

Focused image without motion error

In [ ]:
r_res = 3e8 / (2 * bw * oversample) # Range bin size in input data

# dealias=True removes range spectrum aliasing
img = torchbp.ops.backprojection_polar_2d(data, grid_polar, fc, r_res, pos, dealias=True, data_fmod=data_fmod)
img = img.squeeze(0) # Removes singular batch dimension
# Backprojection image has spectrum with DC at zero index.
# Shifting the spectrum shifts the DC to center bin.
# This makes the solved phase to have same order as the position vector
# Without shifting of the image, fftshift needs to be applied to
# the solved phase for it to be in the same order as the position vector.
# This doesn't affect the absolute value of the image.
img = torchbp.util.shift_spectrum(img)

img_db = 20*torch.log10(torch.abs(img)).detach()

m = torch.max(img_db)

extent = [*grid_polar["r"], *grid_polar["theta"]]

plt.figure()
plt.imshow(img_db.cpu().numpy().T, origin="lower", vmin=m-40, vmax=m, extent=extent, aspect="auto")
plt.xlabel("Range (m)")
plt.ylabel("Angle (sin radians)");

Create corrupted image

In [ ]:
phase_error = torch.exp(1j*2*torch.pi*torch.linspace(-3, 3, ntheta, dtype=torch.float32, device=device)[None,:]**2)

plt.figure()
plt.plot(torch.angle(phase_error.squeeze()).cpu().numpy())
plt.xlabel("Azimuth sample")
plt.ylabel("Phase error (radians)")

img_corrupted = torch.fft.ifft(torch.fft.fft(img, dim=-1) * phase_error, dim=-1)

plt.figure()
plt.imshow(20*torch.log10(torch.abs(img_corrupted)).cpu().numpy().T, origin="lower", vmin=m-40, vmax=m, extent=extent, aspect="auto")
plt.xlabel("Range (m)")
plt.ylabel("Angle (sin radians)");

Phase gradient autofocus with phase difference estimator

In [ ]:
img_pga, phi = torchbp.autofocus.pga(img_corrupted, estimator="pd")

plt.figure()
plt.imshow(20*torch.log10(torch.abs(img_pga)).cpu().numpy().T, origin="lower", vmin=m-40, vmax=m, extent=extent, aspect="auto")
plt.xlabel("Range (m)")
plt.ylabel("Angle (sin radians)");

plt.figure()
plt.plot(torch.angle(torch.exp(1j*phi)).cpu().numpy())
plt.xlabel("Azimuth samples")
plt.ylabel("Phase error (radians)");

Apply maximum likelihood phase gradient autofocus

In [ ]:
img_pga, phi = torchbp.autofocus.pga(img_corrupted, estimator="ml")

plt.figure()
plt.imshow(20*torch.log10(torch.abs(img_pga)).cpu().numpy().T, origin="lower", vmin=m-40, vmax=m, extent=extent, aspect="auto")
plt.xlabel("Range (m)")
plt.ylabel("Angle (sin radians)");

plt.figure()
plt.plot(torch.angle(torch.exp(1j*phi)).cpu().numpy())
plt.xlabel("Azimuth samples")
plt.ylabel("Phase error (radians)");

Weighted least squares estimator. This is the default estimator. It weights each range bin by its estimated signal-to-clutter ratio, which usually gives the most accurate estimate when the image has both strong point-like targets and clutter. Not much difference in this case because it's clutter and noise free.

In [ ]:
img_pga, phi = torchbp.autofocus.pga(img_corrupted, estimator="wls")

plt.figure()
plt.imshow(20*torch.log10(torch.abs(img_pga)).cpu().numpy().T, origin="lower", vmin=m-40, vmax=m, extent=extent, aspect="auto")
plt.xlabel("Range (m)")
plt.ylabel("Angle (sin radians)");

plt.figure()
plt.plot(torch.angle(torch.exp(1j*phi)).cpu().numpy())
plt.xlabel("Azimuth samples")
plt.ylabel("Phase error (radians)");

The solved phase is only meaningful over the occupied part of the azimuth spectrum. The width of the spectrum is set by the aperture length and since the theta axis is oversampled, the spectrum only fills a part of the full azimuth extent (it is centered because of the `shift_spectrum` call above). The phase error and the PGA correction multiply this spectrum, so only the phase over the occupied center bins has any effect on the image. Away from the spectrum there is no signal constraining the estimate. By default `pga` measures the spectrum support (`spectrum_support` argument) and restricts the estimators and the linear trend removal to the occupied bins. Without the gating the noise-only bins would corrupt the estimator statistics and bias the trend removal, which would shift the image.

In [ ]:
spectrum = torch.fft.fft(img_corrupted, dim=-1)
spectrum_db = 20*torch.log10(torch.mean(torch.abs(spectrum), dim=0))
m_s = torch.max(spectrum_db).item()

plt.figure()
plt.plot(spectrum_db.cpu().numpy())
plt.ylim(m_s - 60, m_s + 5)
plt.xlabel("Azimuth samples")
plt.ylabel("Azimuth spectrum (dB)");

Multiplying the FFT of the corrupted image with the solved phase and taking inverse FFT gives the focused image. This should be identical to the image returned by `pga`.

In [ ]:
img_focused = torch.fft.ifft(torch.fft.fft(img_corrupted, dim=-1) * torch.exp(-1j*phi), dim=-1)

plt.figure()
plt.imshow(20*torch.log10(torch.abs(img_focused)).cpu().numpy().T, origin="lower", vmin=m-40, vmax=m, extent=extent, aspect="auto")
plt.xlabel("Range (m)")
plt.ylabel("Angle (sin radians)");

## CFBP + PGA

PGA assumes an image in polar geometry where the azimuth phase error is a multiplicative function of the azimuth spectrum. A Cartesian image, for example from `cfbp` (Cartesian factored backprojection), does not satisfy this assumption, but it can be resampled to a polar grid with `cart_to_polar`, focused with PGA, and resampled back with `polar_to_cart`.

This time the phase error is applied to the raw data with one phase value per sweep, which is how an error from motion measurement inaccuracy would appear.

In [ ]:
nsweeps_cfbp = 512
pos_cfbp = torch.zeros([nsweeps_cfbp, 3], dtype=torch.float32, device=device)
pos_cfbp[:,1] = 0.25 * 3e8 / fc * (torch.arange(nsweeps_cfbp, device=device) - nsweeps_cfbp / 2)

data_cfbp = torchbp.util.generate_fmcw_data(target_pos, target_rcs, pos_cfbp, fc, bw, tsweep, fs)
data_cfbp = torch.fft.ifft(data_cfbp * wr, dim=-1, n=nsamples * oversample)
data_cfbp = data_cfbp * data_fmod_f

# Smooth per-sweep phase error
t = torch.arange(nsweeps_cfbp, device=device) / nsweeps_cfbp
phase_error_sweep = 2 * torch.sin(2 * torch.pi * 2 * t) + 8 * (t - 0.5)**2
data_corrupted = data_cfbp * torch.exp(1j * phase_error_sweep)[:, None]

plt.figure()
plt.plot(pos_cfbp[:,1].cpu().numpy(), phase_error_sweep.cpu().numpy())
plt.xlabel("Aperture position (m)")
plt.ylabel("Phase error (radians)");

Reference and corrupted Cartesian images with CFBP.

In [ ]:
grid_cart = {"x": (90, 110), "y": (-15, 15), "nx": 256, "ny": 384}

img_cart_ref = torchbp.ops.cfbp(data_cfbp, grid_cart, fc, r_res, pos_cfbp, stages=3, data_fmod=data_fmod)[0]
img_cart_corrupted = torchbp.ops.cfbp(data_corrupted, grid_cart, fc, r_res, pos_cfbp, stages=3, data_fmod=data_fmod)[0]

extent_cart = [*grid_cart["x"], *grid_cart["y"]]
m_cart = 20*torch.log10(torch.max(torch.abs(img_cart_ref))).item()

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, img_i, title in zip(axes, (img_cart_ref, img_cart_corrupted), ("Reference", "Corrupted")):
    img_db_i = 20*torch.log10(torch.abs(img_i))
    ax.imshow(img_db_i.cpu().numpy().T, origin="lower", vmin=m_cart-40, vmax=m_cart, extent=extent_cart, aspect="auto")
    ax.set_title(title)
    ax.set_xlabel("y (m)")
axes[0].set_ylabel("x (m)");

Resample the corrupted image to a polar grid centered on the aperture center. `cart_to_polar` removes the range carrier during the interpolation, so the output is equivalent to a dealiased polar image from backprojection. The polar grid needs to cover the Cartesian grid as seen from the origin, theta needs to sample the azimuth bandwidth of the full aperture and r the range envelope bandwidth of the data. `shift_spectrum` centers the azimuth spectrum like before.

In [ ]:
origin_cfbp = torch.mean(pos_cfbp, axis=0)
grid_polar_cfbp = {"r": (89, 112), "theta": (-0.18, 0.18), "nr": 192, "ntheta": 256}

img_polar = torchbp.ops.cart_to_polar(img_cart_corrupted, origin_cfbp, grid_cart, grid_polar_cfbp, fc, method=("lanczos", 8))[0]
img_polar = torchbp.util.shift_spectrum(img_polar)

extent_polar = [*grid_polar_cfbp["r"], *grid_polar_cfbp["theta"]]
img_polar_db = 20*torch.log10(torch.abs(img_polar))

plt.figure()
plt.imshow(img_polar_db.cpu().numpy().T, origin="lower", vmin=m_cart-40, vmax=m_cart, extent=extent_polar, aspect="auto")
plt.xlabel("Range (m)")
plt.ylabel("Angle (sin radians)");

Apply PGA on the polar image with the default settings (weighted least squares estimator with trend removal). The solved phase over the occupied center bins resembles the applied phase error with its linear trend removed. The trend is unobservable to PGA and removing it keeps the image from shifting.

In [ ]:
img_polar_pga, phi = torchbp.autofocus.pga(img_polar)

plt.figure()
plt.plot(torch.angle(torch.exp(1j*phi)).cpu().numpy())
plt.xlabel("Azimuth samples")
plt.ylabel("Solved phase error (radians)");

Undo the spectrum shift and resample the focused polar image back to the Cartesian grid. The shift needs to be undone because `polar_to_cart` interpolates the image and the shift modulates the image at the azimuth Nyquist frequency, which would make it too high frequency to interpolate.

In [ ]:
img_polar_pga = torchbp.util.shift_spectrum(img_polar_pga)
img_cart_pga = torchbp.ops.polar_to_cart(img_polar_pga, origin_cfbp, grid_polar_cfbp, grid_cart, fc, method=("lanczos", 8))[0]

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, img_i, title in zip(axes, (img_cart_corrupted, img_cart_pga), ("Corrupted", "PGA focused")):
    img_db_i = 20*torch.log10(torch.abs(img_i))
    ax.imshow(img_db_i.cpu().numpy().T, origin="lower", vmin=m_cart-40, vmax=m_cart, extent=extent_cart, aspect="auto")
    ax.set_title(title)
    ax.set_xlabel("y (m)")
axes[0].set_ylabel("x (m)");